# Heart Disease Risk — EDA & Model Training

**Course:** MLOps – AIMLCZG523, BITS Pilani WILP  
**Assignment:** 01  

This notebook walks through:
1. Data loading & cleaning
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. Model training & evaluation (Logistic Regression, Random Forest, Gradient Boosting)
5. MLflow experiment tracking summary

> **Note:** The production pipeline runs via `python -m src.train`. This notebook is for *exploration and reporting only*.

In [ ]:
import sys
sys.path.insert(0, '..')  # ensure src/ and api/ are importable from notebooks/

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Libraries loaded')

## 1. Data Loading & Cleaning

In [ ]:
from src.data import load_clean, TARGET, COLUMNS

df = load_clean(save=False)   # load + clean without re-saving
print(f'Shape: {df.shape}')
print(f'\nMissing values:')
print(df.isna().sum()[df.isna().sum() > 0])
print(f'\nTarget distribution:')
print(df[TARGET].value_counts())
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', axis=0)

## 2. Exploratory Data Analysis

### 2.1 Class Balance

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df[TARGET].map({0: 'No Disease', 1: 'Disease'}).value_counts()
bars = ax.bar(counts.index, counts.values, color=['#4C9F70', '#D1495B'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Balance (303 patients)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Patients')
ax.set_ylim(0, 200)
plt.tight_layout()
plt.savefig('../reports/figures/class_balance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Disease prevalence: {counts["Disease"]/counts.sum()*100:.1f}%')

### 2.2 Numeric Feature Distributions by Target

In [ ]:
from src.features import NUMERIC, BINARY, CATEGORICAL, FEATURES

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flat, NUMERIC):
    for label, color in [(0, '#4C9F70'), (1, '#D1495B')]:
        subset = df[df[TARGET] == label][col].dropna()
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label='No Disease' if label == 0 else 'Disease', density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
axes.flat[-1].axis('off')
plt.suptitle('Numeric Feature Distributions by Target', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/numeric_histograms.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

# Top correlations with target
print('\nTop correlations with target:')
print(corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(8))

### 2.4 Disease Rate by Categorical Features

In [ ]:
fig, axes = plt.subplots(1, len(CATEGORICAL), figsize=(20, 4))
for ax, col in zip(axes, CATEGORICAL):
    disease_rate = df.groupby(col)[TARGET].mean()
    disease_rate.plot.bar(ax=ax, color='#30638E', edgecolor='white', rot=0)
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_ylabel('Disease Rate')
    ax.set_ylim(0, 1)
    ax.axhline(df[TARGET].mean(), color='red', linestyle='--', linewidth=1, label='Overall mean')
    ax.legend(fontsize=7)
plt.suptitle('Heart Disease Rate by Categorical Feature', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/categorical_disease_rate.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.5 Key Insights

In [ ]:
# Box plots for top discriminating numeric features
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
top_feats = ['thalach', 'oldpeak', 'age']
for ax, feat in zip(axes, top_feats):
    df.boxplot(column=feat, by=TARGET, ax=ax,
               boxprops=dict(color='#30638E'),
               medianprops=dict(color='#D1495B', linewidth=2))
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Target (0=No Disease, 1=Disease)')
    plt.sca(ax)
    plt.title(feat)
plt.suptitle('Key Feature Distributions by Target', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

The `src/features.py` module defines the complete preprocessing pipeline:

| Group | Features | Transformation |
|-------|----------|----------------|
| **Numeric** | age, trestbps, chol, thalach, oldpeak | Median impute → StandardScaler |
| **Binary** | sex, fbs, exang | Most-frequent impute → passthrough |
| **Categorical** | cp, restecg, slope, ca, thal | Most-frequent impute → OneHotEncoder(handle_unknown='ignore') |

> **Key design decision:** Missing values (`ca`, `thal`) are imputed *inside* the sklearn Pipeline, not in the cleaning step. This guarantees the exact same transformation runs at training and serving time.

In [ ]:
from src.features import build_preprocessor, FEATURES
from sklearn.model_selection import train_test_split

SEED = 42
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train target balance: {y_train.value_counts().to_dict()}')
print(f'Test  target balance: {y_test.value_counts().to_dict()}')
print(f'\nFeatures ({len(FEATURES)}): {FEATURES}')

## 4. Model Training & Evaluation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

CANDIDATES = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, random_state=SEED),
        {'clf__C': [0.01, 0.1, 1, 10], 'clf__class_weight': [None, 'balanced']}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=SEED),
        {'clf__n_estimators': [200, 400], 'clf__max_depth': [None, 5, 8],
         'clf__min_samples_leaf': [1, 3]}
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=SEED),
        {'clf__n_estimators': [100, 200], 'clf__learning_rate': [0.05, 0.1],
         'clf__max_depth': [2, 3]}
    ),
}

results = {}
best_models = {}

for name, (clf, grid) in CANDIDATES.items():
    pipe = Pipeline([('prep', build_preprocessor()), ('clf', clf)])
    search = GridSearchCV(pipe, grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
    search.fit(X_train, y_train)
    
    best = search.best_estimator_
    best_models[name] = best
    
    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'CV ROC-AUC': f"{search.best_score_:.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, y_pred):.4f}",
        'Precision': f"{precision_score(y_test, y_pred):.4f}",
        'Recall': f"{recall_score(y_test, y_pred):.4f}",
        'F1': f"{f1_score(y_test, y_pred):.4f}",
        'Test ROC-AUC': f"{roc_auc_score(y_test, y_proba):.4f}",
        'Best Params': search.best_params_
    }
    print(f'{name:25s}  CV-AUC={search.best_score_:.4f}  Test-AUC={roc_auc_score(y_test, y_proba):.4f}  Recall={recall_score(y_test, y_pred):.4f}')

print('\nTraining complete!')

### 4.1 Results Summary

In [ ]:
summary = pd.DataFrame(results).T.drop(columns=['Best Params'])
summary.index.name = 'Model'
summary.style.highlight_max(color='lightgreen', axis=0)

### 4.2 ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#30638E', '#4C9F70', '#D1495B']
for (name, model), color in zip(best_models.items(), colors):
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name, color=color)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 4.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, best_models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, ax=ax, cmap='Blues',
        display_labels=['No Disease', 'Disease']
    )
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices on Test Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.4 Feature Importance (Random Forest)

In [ ]:
rf_model = best_models['Random Forest']
prep = rf_model.named_steps['prep']
clf = rf_model.named_steps['clf']

# Get feature names after one-hot encoding
ohe_features = prep.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(CATEGORICAL)
all_feature_names = NUMERIC + BINARY + list(ohe_features)

importances = clf.feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot.barh(ax=ax, color='#30638E')
ax.set_title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Model Selection Rationale

| Model | CV ROC-AUC | Selected? | Reason |
|-------|-----------|-----------|--------|
| Logistic Regression | **0.904** | ✅ **Yes** | Highest CV-AUC. Interpretable coefficients. Recall ≥ 0.92 means fewer missed disease cases. |
| Random Forest | 0.898 | No | Marginally lower CV-AUC, more compute. |
| Gradient Boosting | 0.875 | No | Lowest CV-AUC among the three. |

**Why CV-AUC for selection?** Using the *cross-validated* score on the training set avoids leaking the test set into model selection. The test set is used only for final reporting.

**Why prioritise Recall?** A false negative (missing a heart disease case) is clinically costlier than a false alarm. All three models achieve Recall = 0.929 on the test set, so the tiebreaker is CV-AUC.

The best model is saved as `models/model.joblib` — a full sklearn Pipeline (preprocessing + classifier) that can be loaded with a single `joblib.load()` call.

## 6. MLflow Experiment Tracking Summary

In [ ]:
import mlflow
from pathlib import Path

ROOT = Path('..').resolve()
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")

runs = mlflow.search_runs(experiment_names=['heart-disease-classification'])
cols = ['tags.mlflow.runName', 'metrics.cv_roc_auc', 'metrics.test_roc_auc',
        'metrics.test_recall', 'metrics.test_f1', 'metrics.test_accuracy']
available = [c for c in cols if c in runs.columns]
display_df = runs[available].sort_values('metrics.cv_roc_auc', ascending=False)
display_df.columns = [c.replace('tags.mlflow.runName','Model').replace('metrics.','') for c in available]
display_df.style.background_gradient(cmap='Greens', subset=[c for c in display_df.columns if 'auc' in c.lower()])

In [ ]:
# Best model info
import joblib
best = joblib.load(ROOT / 'models' / 'model.joblib')
print(f'Best model type: {type(best.named_steps["clf"]).__name__}')
print(f'Pipeline steps: {list(best.named_steps.keys())}')
print(f'\nBest params: {best.named_steps["clf"].get_params()}')